# Banyan City — free anime rendering on Kaggle

Renders a node's `shots.md` prompts into per-beat clips with **AnimateDiff** on an anime-tuned
**SD1.5** checkpoint, on Kaggle's free GPU quota (30 h/week) — the tree's permanent $0 rendering
floor, reproducible by any citizen (**compute-as-watering**, see `WATERING.md`).

**Setup:** Kaggle → New Notebook → File → Import Notebook → this file. Settings: Accelerator =
**GPU T4 x2**, Internet = ON, phone-verified account. Or drive it headless from a laptop with
`python3 pipeline/kaggle/run_remote.py push <node>`, which is the only mode whose output can be
retrieved — an interactive session dies with the browser tab and `kernels output` 404s.

**Why not Wan 2.1** (tried first, 2026-07-25/26, six pushes): Wan is trained in **bfloat16**, and no
Kaggle free accelerator supports bf16 — T4 is Turing sm_75, P100 is Pascal sm_60, and bf16 arrives
with Ampere sm_80. In fp16 its activations overflow to NaN and every frame decodes to flat grey, at
29 minutes a shot. fp32 is numerically safe and ~8x slower, which cannot finish an episode inside a
12-hour session. AnimateDiff on SD1.5 is fp16-native, runs in minutes, and an anime checkpoint is a
better match for `style.md`'s flat cel-shaded look than a general-purpose video model.

**Hard-won details, each of which cost a run:**
- The repo is cloned to `/kaggle/tmp`, NOT `/kaggle/working`. Everything in `/kaggle/working` becomes
  the session's published output, and Kaggle caps how many files it indexes — a checkout there
  crowded the actual clips out of the output entirely.
- `machine_shape: NvidiaTeslaT4` is set in `kernel-metadata.json`. Without it the batch scheduler
  hands out a P100, which current torch ships no kernels for at all.
- The setup cell defines `transformers.utils.FLAX_WEIGHTS_NAME`, which the batch image's newer
  transformers removed and diffusers 0.33 imports at module load.
- Every clip's middle frame is checked for contrast before it is written, and the session ABORTS on a
  blank one. A numerically dead generation still writes a valid mp4 that passes every container,
  duration and audio check.

Output: `/kaggle/working/clips.zip`, re-zipped after every clip → feed to
`pipeline/render_t3.py <genome> <node> --clips <dir>`. Clips are short (3s at 8fps); `render_t3`
ping-pong-loops them to fill a beat, so a beat never shows a hard loop seam.

Provenance: every clip gets a `meta.yaml` (§7.2). The season's canon quality bar is decided by the
founder on material (R4/D8).


In [ ]:
# ---- config: what to render ----------------------------------------------
GENOME = "sapling"
NODE   = "001"        # any node id with a shots.md
BEATS  = [1]          # e.g. [1, 3] or None for all beats without status ✅
SEED   = 20260719      # fixed base seed: beat N renders with SEED + N (reproducible)
FRAMES = 16            # the v1.5 motion module's NATIVE frame count; 16 @ 8fps = 2s,
                       # and render_t3 ping-pong-loops it to fill a beat
FPS    = 8             # AnimateDiff v1.5 motion module is trained at 8fps
STEPS  = 25            # 30 = faster/rougher, 50 = slower/cleaner
# SD1.5 anime checkpoints, tried in order — the first that loads anonymously wins.
# A gated repo returns 401 without a HuggingFace token, and a token is a
# credential (founder-reserved), so the notebook only ever uses open weights.
# Linaqruf/anything-v3.0 was the first choice and is gated.
BASES  = ["Lykon/dreamshaper-8",                          # illustrative, AnimateDiff-friendly
          "stable-diffusion-v1-5/stable-diffusion-v1-5",   # PROVEN to animate (spread 186)
          "gsdf/Counterfeit-V2.5"]                         # stills fine, does NOT animate
ADAPT  = "guoyww/animatediff-motion-adapter-v1-5-3"     # AnimateDiff motion module
REPO_URL = "https://github.com/olegmlkvorg/banyan-city.git"


In [ ]:
# ---- setup: deps + repo (canon prompts come from shots.md, not a paste) ---
# Do NOT reinstall torch. Kaggle ships a working torch/torchvision pair; the
# first real run (2026-07-25) tried to pin its own and hit INTERNAL ASSERT
# FAILED in Dtype.cpp — a fresh torchvision against the already-imported
# torch. Install diffusers with --no-deps so pip cannot pull a second torch in
# behind it. WanPipeline needs diffusers >= 0.33 (0.32 lacks it entirely —
# that was the failure after the pin).
%pip -q install --no-deps "diffusers==0.33.1"
%pip -q install ftfy imageio imageio-ffmpeg pyyaml psutil

# A kernel that already imported an older diffusers keeps it in memory no
# matter what pip writes to disk (this bit the founder on 2026-07-25: the
# session still held 0.32.2). Compare disk vs memory and say so plainly.
import sys
from importlib.metadata import version
on_disk = version("diffusers")
in_mem = getattr(sys.modules.get("diffusers"), "__version__", None)
print(f"diffusers on disk: {on_disk}" + (f" | already imported in this kernel: {in_mem}" if in_mem else ""))
if in_mem and in_mem != on_disk:
    print("\n*** STOP: this kernel is holding an older diffusers.\n"
          "    Run > Restart & clear cell outputs, then Run All again.\n"
          "    (Nothing is lost — finished clips are skipped on re-run.) ***\n")


# Kaggle's BATCH image ships a newer transformers than its interactive one, and
# `transformers.utils.FLAX_WEIGHTS_NAME` is gone from it. diffusers 0.33 imports
# that name at module load, so `from diffusers import WanPipeline` died with
# "cannot import name 'FLAX_WEIGHTS_NAME'" before touching the GPU (first batch
# push, 2026-07-25). The names are plain filename constants and nothing on the
# Wan path reads a flax/tf checkpoint, so define what is missing rather than
# repinning transformers — a repin drags tokenizers and risks the torch pair.
import transformers.utils as _tu
for _name, _val in (("FLAX_WEIGHTS_NAME", "flax_model.msgpack"),
                    ("TF2_WEIGHTS_NAME", "tf_model.h5"),
                    ("TF_WEIGHTS_NAME", "model.ckpt")):
    if not hasattr(_tu, _name):
        setattr(_tu, _name, _val)
        print(f"shimmed transformers.utils.{_name} (removed upstream)")

import pathlib
import subprocess
import sys

# Clone OUTSIDE /kaggle/working. Everything in /kaggle/working becomes the
# session's downloadable output, so cloning the repo there put the whole
# checkout — every committed episode mp4 included — into the output: the first
# successful run's `kernels output` was still pulling at 755 MB and had not
# reached the one clip that was actually rendered (2026-07-25). /kaggle/tmp is
# scratch and is not published.
CHECKOUT = pathlib.Path("/kaggle/tmp/banyan-city")
if not CHECKOUT.exists():
    CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT / "pipeline"))
from generate_shots import parse_shots
from sd_prompt import compress  # SD1.5's text encoder stops at 77 tokens
import yaml

node_dirs = [d for d in (CHECKOUT / "genomes" / GENOME / "nodes").iterdir() if d.is_dir()]
node_dir = next((d for d in sorted(node_dirs) if d.name.startswith(NODE)), None)
assert node_dir, f"no node dir starting with {NODE!r} — check NODE above"
shots = parse_shots((node_dir / "shots.md").read_text())
todo = [s for s in shots if (BEATS is None and not s["done"]) or (BEATS and s["num"] in BEATS)]
print(f"{len(todo)} beat(s) to render for {node_dir.name}:")
for s in todo:
    print(f"  {s['num']:02d} {s['slug']}")

# the negative prompt is defined here so the model cell's bisect can use it too
NEG = ("photorealistic, 3d render, text, watermark, signature, low quality, blurry, "
       "extra limbs, deformed, jpeg artifacts, realistic skin texture")


In [ ]:
# ---- model: AnimateDiff on an SD1.5 checkpoint that PROVES it animates --------
# Why not Wan 2.1: trained in bfloat16, and no Kaggle free accelerator has bf16
# (T4 is sm_75, P100 is sm_60; bf16 starts at Ampere sm_80). In fp16 it overflows
# to NaN and every frame decodes flat grey.
#
# And why this cell probes instead of just loading. On 2026-07-26, sixteen renders
# came back blank and every component was blamed in turn. Bisecting settled it:
#
#   plain still, Counterfeit-V2.5, no adapter ......... luma spread  57  (fine)
#   AnimateDiff on vanilla SD1.5 ..................... luma spread 186  (vivid)
#   AnimateDiff on Counterfeit-V2.5 .................. luma spread  16  (dead)
#
# The adapter is fine and the checkpoint renders stills fine; the PAIRING is
# broken. Nothing about loading either one reports that, so the only honest test
# is to animate a few frames and look at them. That is what this does — a short
# 8-frame probe per candidate, first one that produces an actual picture wins.
# A few minutes here buys back the hours a wrong pairing costs.
import gc

import numpy as np
import psutil
import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator = GPU (needs phone verification)"
_cap = torch.cuda.get_device_capability(0)
_have = torch.cuda.get_arch_list()
if f"sm_{_cap[0]}{_cap[1]}" not in _have:
    raise SystemExit(
        f"{torch.cuda.get_device_name(0)} is sm_{_cap[0]}{_cap[1]}, and this torch "
        f"was built for {_have}. Set Accelerator to GPU T4 x2 (sm_75).")

from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
from diffusers.utils import export_to_video

vram = torch.cuda.get_device_properties(0).total_memory / 2**30
ram = psutil.virtual_memory().available / 2**30
print(f"{torch.cuda.get_device_name(0)}: {vram:.1f} GiB VRAM | {ram:.1f} GiB RAM available")

PROBE_FLOOR = 35.0   # dead grey runs 14-22; real footage in this style runs 52-144


def luma_spread(img):
    """LUMA spread, 0-255. Must be luma: RGB percentiles are inflated by colour,
    and a blank frame reads 28 in RGB against 18 in luma."""
    a = np.asarray(img, dtype=np.float32)
    if not np.isfinite(a).all():
        return 0.0
    if a.ndim == 3 and a.shape[-1] >= 3:
        a = a[..., 0] * 0.299 + a[..., 1] * 0.587 + a[..., 2] * 0.114
    lo, hi = np.percentile(a, 10), np.percentile(a, 90)
    return float(hi - lo) * (255.0 if a.max() <= 1.001 else 1.0)


def build(base):
    adapter = MotionAdapter.from_pretrained(ADAPT, torch_dtype=torch.float16)
    pipe = AnimateDiffPipeline.from_pretrained(base, motion_adapter=adapter,
                                              torch_dtype=torch.float16)
    # Do NOT force beta_schedule. AnimateDiff's README uses "linear" because it
    # demos against vanilla SD1.5; anime finetunes are "scaled_linear", and forcing
    # the wrong variance curve denoises against the wrong noise schedule.
    pipe.scheduler = DDIMScheduler.from_config(
        pipe.scheduler.config, clip_sample=False, timestep_spacing="linspace",
        steps_offset=1)
    pipe.enable_vae_slicing()
    pipe.to("cuda")
    return pipe


probe_prompt = compress(todo[0]["prompt"])[0]
pipe = BASE = None
for cand in BASES:
    try:
        cp = build(cand)
        fr = cp(prompt=probe_prompt, negative_prompt=NEG, height=512, width=512,
                num_frames=8, num_inference_steps=12, guidance_scale=7.5,
                generator=torch.Generator(device="cpu").manual_seed(SEED)).frames[0]
        sp = float(np.median([luma_spread(f) for f in fr]))
        fr[len(fr) // 2].save(f"/kaggle/working/PROBE-{cand.split('/')[-1]}.png")
        print(f"probe {cand}: luma spread {sp:.0f} "
              + ("ANIMATES" if sp >= PROBE_FLOOR else "-> DEAD, skipping"), flush=True)
        if sp >= PROBE_FLOOR:
            pipe, BASE = cp, cand
            break
        del cp, fr
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f"probe {cand}: unavailable ({type(e).__name__}: {str(e)[:80]})", flush=True)
        gc.collect(); torch.cuda.empty_cache()

if pipe is None:
    raise SystemExit(
        f"none of {BASES} produced a moving picture. The adapter itself is fine "
        "(vanilla SD1.5 measured 186 on 2026-07-26), so this is a pairing or a "
        "diffusers-version problem — do not spend a session rendering.")

gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"pipeline ready — AnimateDiff on {BASE}; "
      f"{{'unet': {pipe.unet.dtype}, 'vae': {pipe.vae.dtype}}}; "
      f"{free/2**30:.1f} of {total/2**30:.1f} GiB VRAM free")


In [ ]:
# ---- generate: 9:16 vertical, FRAMES @ FPS -----------------------------------
# Re-zip after every clip (a killed kernel never reaches the packing cell) and
# print per-shot minutes so the remote log says whether the queue can finish.
import shutil
import time
from datetime import date
import numpy as np

NEG = ("photorealistic, 3d render, text, watermark, signature, low quality, blurry, "
       "extra limbs, deformed, jpeg artifacts, realistic skin texture")
out = pathlib.Path("/kaggle/working/clips"); out.mkdir(parents=True, exist_ok=True)

for s in todo:
    dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
    if dest.exists():
        print(f"skip {dest.name} (exists)"); continue
    t0 = time.time()
    print(f"beat {s['num']:02d} ({s['slug']}) …", flush=True)
    g = torch.Generator(device="cpu").manual_seed(SEED + s["num"])

    # CLIP stops at 77 tokens and shots.md prompts run 113-145, so the raw prompt
    # loses its action and keeps only style words. compress() puts a short style
    # tag first and the action second, and drops the "no photorealism / no text"
    # tail that the negative prompt already covers.
    ptext, dropped = compress(s["prompt"])
    print(f"   prompt: {ptext}", flush=True)
    if dropped:
        print(f"   (dropped, too long: {' '.join(dropped)[:120]})", flush=True)

    # Instrument the latents so the NEXT failure names its own cause instead of
    # costing another guess: if the latents are healthy and the frames are dead,
    # the VAE is at fault; if the latents are already degenerate, it is the UNet
    # or the schedule.
    def _peek(pipe_, step, timestep, kw):
        if step in (0, STEPS // 2, STEPS - 1):
            lat = kw.get("latents")
            if lat is not None:
                f = lat.float()
                print(f"   step {step:2d} latents: mean {f.mean():+.3f} std {f.std():.3f} "
                      f"min {f.min():+.2f} max {f.max():+.2f} "
                      f"nan {int(torch.isnan(f).sum())}", flush=True)
        return kw

    def generate(h, w, n):
        return pipe(prompt=ptext, negative_prompt=NEG, height=h, width=w,
                    num_frames=n, num_inference_steps=STEPS, guidance_scale=7.5,
                    generator=g, callback_on_step_end=_peek).frames[0]

    # 512x512 is the v1.5 motion module's native size. 432x768 — which this cell
    # used until 2026-07-26 — is 2.5x the pixels and produced washed-out mush on
    # every attempt. render_t3 does the 9:16 framing (scale + centre-crop), so
    # generating square costs nothing.
    try:
        frames = generate(512, 512, FRAMES)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if not isinstance(e, torch.cuda.OutOfMemoryError) and \
           not any(k in str(e) for k in ("CUBLAS", "out of memory", "CUDA error")):
            raise
        print(f"   {type(e).__name__} — retrying smaller", flush=True)
        gc.collect(); torch.cuda.empty_cache()
        frames = generate(640, 360, max(16, FRAMES - 8))

    # Verify the PIXELS, not the exit code. A generation can fail numerically and
    # still write a perfectly valid mp4: Wan did exactly that for 28.9 minutes a
    # shot and every container/audio check passed on five seconds of grey.
    #
    # Calibrated against the 31 archived clips of this exact anime style rather
    # than guessed: their per-frame luma spread (90th percentile minus 10th) runs
    # a median of 52-144, median-of-minimums 88. The grey Wan renders measured 14
    # and 22. So the floor sits at 35 — comfortably above dead grey, comfortably
    # below real pastel cel-shading, which this style deliberately keeps soft.
    #
    # Judged on the MEDIAN across frames, not one frame: real clips dip to a
    # single frame of 35-40 while their medians sit far higher, so a one-frame
    # check produces false alarms. And the clip is WRITTEN either way — aborting
    # without an artifact left nothing to look at, which is how a spread of 39
    # became unanswerable instead of merely suspicious.
    def spread_of(fr):
        """LUMA spread, 0-255. Must be luma, not RGB.

        The threshold below is calibrated on the archived clips' LUMA spread, and
        this measured RGB — percentiles across all three channels, which colour
        alone inflates. On 2026-07-26 a blank frame read 28 in RGB and 18 in luma
        against a floor of 35, so the guard passed a grey clip while qa_episode
        failed the same file. Comparing two different quantities is not a
        threshold problem; it is measuring the wrong thing."""
        a = np.asarray(fr, dtype=np.float32)
        if not np.isfinite(a).all():
            return 0.0
        if a.ndim == 3 and a.shape[-1] >= 3:
            a = a[..., 0] * 0.299 + a[..., 1] * 0.587 + a[..., 2] * 0.114
        lo, hi = np.percentile(a, 10), np.percentile(a, 90)
        return float(hi - lo) * (255.0 if a.max() <= 1.001 else 1.0)

    spreads = [luma_spread(f) for f in frames[::max(1, len(frames) // 8)]]
    spread = float(np.median(spreads))
    export_to_video(frames, str(dest), fps=FPS)
    if spread < 35:
        suspect = dest.with_suffix(".SUSPECT.mp4")
        dest.rename(suspect)
        shutil.make_archive("/kaggle/working/clips", "zip", out)
        raise SystemExit(
            f"beat {s['num']:02d} looks BLANK: median luma spread {spread:.0f}, "
            f"per-sample {[round(x) for x in spreads]}. Real footage in this style "
            f"runs 52-144; dead grey runs 14-22. Written to {suspect.name} so it can "
            "be looked at. Stopping rather than spending the session on grey.")
    shutil.make_archive("/kaggle/working/clips", "zip", out)
    took = time.time() - t0
    left = len([x for x in todo if not (out / f"{x['num']:02d}-{x['slug']}.mp4").exists()])
    print(f"   {dest.name} in {took/60:.1f} min, contrast {spread:.0f} ({left} left)", flush=True)

    dest.with_suffix(".meta.yaml").write_text(
        "# Shot provenance (\u00a77.2)\n" + yaml.safe_dump({
            "platform": "kaggle-free-gpu", "model": f"AnimateDiff {ADAPT} on {BASE}",
            "prompt": ptext, "prompt_source": s["prompt"], "negative_prompt": NEG, "seed": SEED + s["num"],
            "steps": STEPS, "frames": len(frames), "fps": FPS,
            "cost_usd": 0.00, "generated": str(date.today()),
        }, sort_keys=False))


In [ ]:
# ---- pack for download ------------------------------------------------------
import shutil
shutil.make_archive("/kaggle/working/clips", "zip", "/kaggle/working/clips")
print("download clips.zip from the Output tab, then locally:")
print(f"  python3 pipeline/render_t3.py {GENOME} {NODE} --clips <unzipped-dir> --out /tmp/{NODE}-episode.mp4")